In [ ]:
import subprocess, sys, os, time, glob, shutil, hashlib, json
T0 = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java")
sh(f"{sys.executable} -m pip install -q 'transformers==5.20.2' 'peft==0.21.2' 'trl==0.29.1' accelerate pycocoevalcap sacrebleu bert_score 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y -q torchao")
r = subprocess.run([sys.executable, "-c", "import trl, transformers, peft; print(trl.__version__, transformers.__version__, peft.__version__)"], capture_output=True, text=True)
print(r.stdout, r.stderr[-500:], flush=True)
assert r.stdout.split() == ["0.29.1", "5.20.2", "0.21.2"], "DUNG: lech phien ban"
W = "/kaggle/working"; S = f"{W}/s"; OUT = f"{W}/out"
os.makedirs(S, exist_ok=True); os.makedirs(OUT, exist_ok=True)
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
MD5 = {'grpo_luoi.py': '92d0a9bc6b8cae8a480aef08f99fb9d8', 'grpo_spice.py': '8489108147c8d1ed80c3dc67c8ad3d3f', 'build_branch_data.py': '619e63e123a6dbf60086e65ee94a3912', 'grpo_codien.py': '5499d6041bf0fb7c780099abcaa27f32', 'grpo_ppo.py': 'e3330d4469f6b60c9e3795e2d46b83db', 'grpo_moi_b1.py': 'f0f83534642620c738353bbc3d86c27e', 'grpo_moi_b2.py': '821411b88e1f18fc8d1f3062e5e21f06', 'thuong_moi.py': 'dc88473e9115f8bcc954fb402f43a368', 'metric_exec.py': '9bf0b84145458fd55919a5e161b9766f', 'descriptors_ten.jsonl': 'a3535c64534cc88f923cb36da102d91e', 'grpo_moi_b4.py': 'e5d2443aa4cd1e05fb04d441ded5c4e7', 'thuong_t6.py': '64d0caf482fb56903b706255eb7c265c'}
for f, h in MD5.items():
    k = [p for p in glob.glob(f"/kaggle/input/**/{f}", recursive=True) if "/notebooks/" not in p]
    if len(k) > 1: k = [p for p in k if md5(p) == MD5[f]]
    assert len(k) == 1, f"DUNG: can dung mot {f}, thay {k}"
    shutil.copyfile(k[0], f"{S}/{f}"); assert md5(f"{S}/{f}") == h, f"DUNG: {f} lech ban may nha"
open(f"{S}/thu.py", "w").write('"""python thu.py <file script> <cờ...>: chạy script như __main__ nhưng nạp mô hình tí hon."""\nimport os, sys, runpy\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport torch\nimport grpo_spice as GS\nfrom transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration\nTINY = "trl-internal-testing/tiny-Qwen2_5_VLForConditionalGeneration"\nclass AnyEq(int):\n    def __eq__(self, o): return True\n    __hash__ = int.__hash__\ndef nap(a):\n    return AutoProcessor.from_pretrained(TINY), Qwen2_5_VLForConditionalGeneration.from_pretrained(TINY, dtype=torch.float32), torch.float32\nGS.nap = nap\nGS._n_s1 = lambda p: AnyEq(0)\nf = sys.argv[1]\nsys.argv = sys.argv[1:]\nrunpy.run_path(f, run_name="__main__")\n')
import importlib.util
SPL = list(importlib.util.find_spec("pycocoevalcap").submodule_search_locations)[0] + "/spice/lib"
for j, sz in {"stanford-corenlp-3.6.0.jar": 7270741, "stanford-corenlp-3.6.0-models.jar": 378116526}.items():
    src = glob.glob(f"/kaggle/input/**/{j}", recursive=True); assert src, f"DUNG: thieu {j}"
    shutil.copyfile(src[0], f"{SPL}/{j}"); assert os.path.getsize(f"{SPL}/{j}") == sz
# bundle nhỏ: dữ liệu thật (câu nhắc, câu chuẩn, OCR), ảnh xám 112x224, adapter giả r=8
REAL = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "images" in d)
B = "/tmp/bundle_thu"   # ngoài /kaggle/working để Output không kéo 4.000 ảnh
os.makedirs(f"{B}/adapter_s1_seed101", exist_ok=True)
for f in ("p1_train_rows.jsonl", "p1_val_rows.jsonl", "ocr.jsonl"):
    shutil.copyfile(f"{REAL}/{f}", f"{B}/{f}")
from PIL import Image
for l in open(f"{B}/p1_train_rows.jsonl"):
    p = f"{B}/" + json.loads(l)["image"]; os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p): Image.new("RGB", (112, 224), (200, 200, 200)).save(p)
json.dump({"r": 8, "lora_alpha": 16, "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"]}, open(f"{B}/adapter_s1_seed101/adapter_config.json", "w"))
print("chuan bi xong", round(time.time() - T0), "s", flush=True)


In [ ]:
open(f"{S}/thu_rng.py", "w").write('"""python thu_rng.py <grpo_spice.py|grpo_luoi.py> <cờ...>"""\nimport os, sys, json, hashlib\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport torch, trl, transformers, peft\nimport grpo_spice as GS\nfrom transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration\nTINY = "trl-internal-testing/tiny-Qwen2_5_VLForConditionalGeneration"\nclass AnyEq(int):\n    def __eq__(self, o): return True\n    __hash__ = int.__hash__\ndef nap(a):\n    return AutoProcessor.from_pretrained(TINY), Qwen2_5_VLForConditionalGeneration.from_pretrained(TINY, dtype=torch.float32), torch.float32\nGS.nap = nap\nGS._n_s1 = lambda p: AnyEq(0)\ntorch.cuda.reset_peak_memory_stats = lambda *a, **k: None   # grpo_spice.py in VRAM, CPU không có\ntorch.cuda.max_memory_allocated = lambda *a, **k: 0\ntorch.cuda.get_device_name = lambda *a, **k: "cpu"\ntorch.cuda.get_device_capability = lambda *a, **k: (0, 0)\n_C = trl.GRPOConfig\nclass Cfg(_C):               # grpo_spice.py đặt fp16 = not bf, CPU không nhận fp16\n    def __post_init__(self):\n        self.fp16 = False; self.bf16 = False; super().__post_init__()\ntrl.GRPOConfig = Cfg\nmd5 = lambda t: hashlib.md5(t.detach().cpu().contiguous().numpy().tobytes()).hexdigest()\n_g = Qwen2_5_VLForConditionalGeneration.generate\ndef gen(self, *a, **k):\n    rs = hashlib.md5(torch.get_rng_state().numpy().tobytes()).hexdigest()\n    ids = k.get("input_ids", a[0] if a else None)\n    out = _g(self, *a, **k)\n    seq = out.sequences if hasattr(out, "sequences") else out\n    rec = dict(script=os.path.basename(sys.argv[1]), versions=[trl.__version__, transformers.__version__, peft.__version__, torch.__version__],\n               rng_truoc=rs, input=md5(ids) if ids is not None else None, out=md5(seq), shape=list(seq.shape),\n               duoi_chuoi_0=seq[0, -12:].tolist(), duoi_chuoi_1=seq[1, -12:].tolist() if seq.shape[0] > 1 else None)\n    print("[rng]", json.dumps(rec), flush=True)\n    json.dump(rec, open(os.environ["RNG_OUT"], "w"))\n    os._exit(0)\nQwen2_5_VLForConditionalGeneration.generate = gen\nf = sys.argv[1]; sys.argv = sys.argv[1:]\nif f == "grpo_spice.py":\n    GS.main()\nelse:\n    import grpo_luoi as GL; GL.main()\nprint("[rng] KHÔNG gặp lần sinh nào", flush=True); sys.exit(3)\n')
# bộ thư viện cũ của ck500 gốc, venv riêng
r = sh(f"{sys.executable} -m pip install -q --no-deps --target /tmp/oldlib 'torch==2.10.0' 'torchvision==0.25.0' 'transformers==5.17.0' 'peft==0.21.1' 'trl==0.29.1' --extra-index-url https://download.pytorch.org/whl/cpu 2>&1 | tail -3")
OLD = {**os.environ, "PYTHONPATH": "/tmp/oldlib"}
r2 = subprocess.run([sys.executable, "-c", "import trl, transformers, peft, torch; print(trl.__version__, transformers.__version__, peft.__version__, torch.__version__)"], capture_output=True, text=True, env=OLD)
print("venv cu:", r2.stdout, r2.stderr[-400:], flush=True)
PY = {"moi": sys.executable}
if r2.returncode == 0 and r2.stdout.split()[:3] == ["0.29.1", "5.17.0", "0.21.1"] and r2.stdout.split()[3].startswith("2.10.0"): PY["cu"] = sys.executable
KQ = {}
for env, py in PY.items():
    for scr in ("grpo_spice.py", "grpo_luoi.py"):
        for lan in (1, 2):
            tag = f"{env}_{scr[:-3]}_{lan}"; o = f"{OUT}/{tag}"; shutil.rmtree(o, ignore_errors=True)
            cmd = [py, "thu_rng.py", scr, "--train", "--no-q4", "--bs", "4", "--accum", "4", "--G", "8", "--reward", "spice",
                   "--seed", "101", "--bundle", B, "--merged", "x", "--out", o, "--max-steps", "2"]
            if scr == "grpo_luoi.py": cmd += ["--algo", "grpo"]
            t = time.time()
            with open(f"{OUT}/{tag}.log", "w") as lg:
                p = subprocess.run(cmd, cwd=S, stdout=lg, stderr=subprocess.STDOUT, text=True,
                                   env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "RNG_OUT": f"{OUT}/{tag}.json", **({"PYTHONPATH": "/tmp/oldlib"} if env == "cu" else {})})
            L = open(f"{OUT}/{tag}.log", errors="ignore").read()
            KQ[tag] = json.load(open(f"{OUT}/{tag}.json")) if os.path.exists(f"{OUT}/{tag}.json") else dict(loi=L[-1500:])
            print(f"{(time.time()-T0)/60:5.1f} ph · {tag} · ma {p.returncode} · {round(time.time()-t)} s", flush=True)
            print("   ", json.dumps(KQ[tag], ensure_ascii=False)[:600], flush=True)
def k(t, f): return KQ.get(t, {}).get(f)
TOM = {}
for env in PY:
    for scr in ("grpo_spice", "grpo_luoi"):
        TOM[f"tat_dinh {env} {scr}"] = k(f"{env}_{scr}_1", "out") == k(f"{env}_{scr}_2", "out") is not None
    for f in ("rng_truoc", "input", "out"):
        TOM[f"{env}: spice == luoi ({f})"] = k(f"{env}_grpo_spice_1", f) == k(f"{env}_grpo_luoi_1", f)
if "cu" in PY:
    for scr in ("grpo_spice", "grpo_luoi"):
        for f in ("rng_truoc", "input", "out"):
            TOM[f"{scr}: cu == moi ({f})"] = k(f"cu_{scr}_1", f) == k(f"moi_{scr}_1", f)
print(json.dumps(TOM, indent=1, ensure_ascii=False), flush=True)
json.dump(dict(KQ=KQ, TOM=TOM), open(f"{OUT}/ket_qua.json", "w"), indent=1, ensure_ascii=False)
shutil.make_archive(f"{W}/kiem_rng", "zip", OUT)
